# Supp. Fig. 13 â€” Correlations: Mongillo model vs neural data

Compares ACG and CCG structure between the Mongillo (2008) attractor model and recorded ephys data.

**Sections:**
1. Trial raster & decoder examples (Mongillo model)
2. Log-odds distributions â€” model
3. Log-odds distributions â€” neural data
4. Bimodality coefficient comparison
5. ACG *(loads precomputed `.npz`)*
6. CCG *(loads precomputed `.npz`)*
7. **[Run once]** Precomputation cells that generate the `.npz` files

**Precomputed files expected in `data_path`:**
- `mongillo_acg_{win}ms.npz`, `mongillo_ccg_{win}ms.npz`
- `ephys_acg.npz`, `ephys_ccg.npz`

Run section 7 once with the raw data files to generate them.

In [ ]:
import math
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from scipy.stats import kurtosis, skew, norm
from scipy.signal import fftconvolve
from matplotlib.ticker import MaxNLocator
from matplotlib.backends.backend_pdf import PdfPages

# Neuroscience packages â€” install with: pip install elephant neo quantities
from quantities import ms
from neo.core import SpikeTrain
from elephant.kernels import GaussianKernel
from elephant.statistics import instantaneous_rate

sys.path.insert(0, str(Path.cwd().parents[2]))
from config import DATA_DIR, FIGURES_OUT

In [ ]:
COLORLEFT  = 'teal'
COLORRIGHT = '#FF8D3F'

data_path    = str(DATA_DIR / 'supp_figures' / 'supp_fig_13_correlations_mogillo_data') + '/'
save_figures = str(FIGURES_OUT / 'supp_figures' / 'supp_fig_13_correlations_mogillo_data') + '/'
os.makedirs(save_figures, exist_ok=True)

win = 150   # 150 or 250 â€” selects which model files to load
r   = 100

---
## 1. Trial raster & decoder examples (Mongillo model)

In [ ]:
# Files: mongillo_model_raster_{win}ms.csv  +  decoder_all_mongillo_window{win}_20_r{r}.csv
dfSP_raster = pd.read_csv(data_path + f'mongillo_model_raster_{win}ms.csv', index_col=0)
dfLL_raster = pd.read_csv(data_path + f'decoder_all_mongillo_window{win}_20_r{r}.csv')

dfSP_raster['spiketime'] /= 1000

dfSP_raster['a_Response_ON'] = dfSP_raster['spiketime'] - 11.35
dfSP_raster['a_Delay_ON']    = dfSP_raster['spiketime'] - 1.35
dfSP_raster['a_Stimulus_ON'] = dfSP_raster['spiketime'] - 1.0

dfLL_raster['a_Response_ON'] = dfLL_raster['time'] - 11.35
dfLL_raster['a_Delay_ON']    = dfLL_raster['time'] - 1.35
dfLL_raster['a_Stimulus_ON'] = dfLL_raster['time'] - 1.0

In [ ]:
def convolve_trial_simple(nx, T, x_min=-1, x_max=10.5, align='Stimulus_ON', sampling=10):
    """Convolve spikes for a single neuron/trial using aligned times (seconds)."""
    times_spikes = nx.loc[nx['trial'] == T, 'a_' + align]
    if times_spikes.empty or len(times_spikes) < 2:
        return None

    times_ms   = times_spikes.values * 1000
    start_time = (x_min * 1000 - 500) * ms
    stop_time  = (x_max * 1000 + 500) * ms

    try:
        spiketrain = SpikeTrain(times_ms, units=ms, t_stop=stop_time, t_start=start_time)
    except ValueError:
        return None

    gaus_rate = instantaneous_rate(
        spiketrain,
        sampling_period=sampling * ms,
        kernel=GaussianKernel(100 * ms)
    )
    times_  = gaus_rate.times.rescale('s').magnitude
    firing  = gaus_rate.magnitude.flatten()

    df_out = pd.DataFrame({'time_centered': times_, 'firing': firing})
    df_out['trial']      = T
    df_out['neuron']     = nx['neuron'].unique()[0]
    df_out['population'] = nx['population'].unique()[0]
    return df_out


def plot_trials_grid_live(
        dfSP, dfLL, trials_list, align='Stimulus_ON', sampling=5,
        save_path=None, filename_base='trials',
        show_neuron_ids=False, ncols=5, show_decoder=True):

    n_trials   = len(trials_list)
    nrows      = math.ceil(n_trials / ncols)
    n_subrows  = 3 if show_decoder else 2
    height_ratios = ([2.5, 1.25, 1.25] if show_decoder else [2.5, 1.25]) * nrows

    fig, axes = plt.subplots(
        nrows * n_subrows, ncols,
        figsize=(ncols * 5.5, nrows * (7 if show_decoder else 6)),
        squeeze=False,
        gridspec_kw={'height_ratios': height_ratios}
    )

    for idx, trial in enumerate(trials_list):
        print(f'Processing trial {trial}...')
        row = idx // ncols
        col = idx % ncols

        ax_raster  = axes[row * n_subrows + 0, col]
        ax_fr      = axes[row * n_subrows + 1, col]
        ax_decoder = axes[row * n_subrows + 2, col] if show_decoder else None

        sp_trial = dfSP[dfSP['trial'] == trial]

        right_neurons = sp_trial[sp_trial['population'] == 1]['neuron'].unique()
        left_neurons  = sp_trial[sp_trial['population'] == 0]['neuron'].unique()
        right_neurons = np.random.choice(right_neurons, min(50, len(right_neurons)), replace=False)
        left_neurons  = np.random.choice(left_neurons,  min(50, len(left_neurons)),  replace=False)
        ordered_neurons = np.concatenate([right_neurons, left_neurons])
        neuron_to_y     = {n: i for i, n in enumerate(ordered_neurons)}
        markersize      = max(1, 30 / np.sqrt(len(ordered_neurons)))

        x_min = sp_trial['a_' + align].min()
        x_max = sp_trial['a_' + align].max()

        # Raster
        for neuron_id in ordered_neurons:
            color  = COLORRIGHT if neuron_id in right_neurons else COLORLEFT
            spikes = sp_trial.loc[sp_trial['neuron'] == neuron_id, 'a_' + align].values
            y      = neuron_to_y[neuron_id]
            ax_raster.plot(spikes, np.repeat(y, len(spikes)),
                           '|', markersize=markersize, markeredgewidth=0.8, color=color)
        if show_neuron_ids:
            ax_raster.set_yticks(list(neuron_to_y.values()))
            ax_raster.set_yticklabels(list(neuron_to_y.keys()), fontsize=6)
        ax_raster.set_ylabel('Units')
        ax_raster.set_ylim(-0.5, len(ordered_neurons))

        # Firing rate
        all_fr = []
        for neuron_id in ordered_neurons:
            nx    = sp_trial[sp_trial['neuron'] == neuron_id]
            fr_df = convolve_trial_simple(nx, trial, align=align, sampling=sampling)
            if fr_df is not None:
                all_fr.append(fr_df)

        if all_fr:
            df_all_fr  = pd.concat(all_fr, ignore_index=True)
            mean_all   = df_all_fr.groupby('time_centered')['firing'].mean()
            mean_right = df_all_fr[df_all_fr['population'] == 1].groupby('time_centered')['firing'].mean()
            mean_left  = df_all_fr[df_all_fr['population'] == 0].groupby('time_centered')['firing'].mean()
            if not mean_right.empty:
                ax_fr.plot(mean_right.index, mean_right.values, color=COLORRIGHT, linewidth=1.5)
            if not mean_left.empty:
                ax_fr.plot(mean_left.index, mean_left.values, color=COLORLEFT,  linewidth=1.5)
            y_max = mean_all.max() + 5
            ax_fr.set_ylim(0, 20)
            ax_fr.yaxis.set_major_locator(MaxNLocator(nbins='auto'))

        ax_fr.set_ylabel('FR (sp/s)')
        ax_fr.set_xlim(x_min, x_max + 1)

        # Decoder
        if show_decoder:
            ll_trial = dfLL[dfLL['trial'] == trial].groupby('time', as_index=False)['loglikelihood'].mean()
            ax_decoder.plot(ll_trial.time, ll_trial['loglikelihood'], color='black', linewidth=1.5)
            ax_decoder.axhline(0, linestyle=':', color='grey')
            ax_decoder.set_ylabel('Log odds')
            ax_decoder.set_ylim(dfLL['loglikelihood'].min() - 0.5, dfLL['loglikelihood'].max() + 0.5)
            ax_decoder.set_xlim(x_min, x_max + 1)
            ax_decoder.set_xticks(np.arange(0, 11, 5))

        # Grey bars (stimulus + response windows)
        for ax in ([ax_raster, ax_fr, ax_decoder] if show_decoder else [ax_raster, ax_fr]):
            ax.axvspan(0,     0.35,  color='grey', alpha=0.3, zorder=0, linewidth=0)
            ax.axvspan(10.35, 10.85, color='grey', alpha=0.3, zorder=0, linewidth=0)

        bottom_ax = ax_decoder if show_decoder else ax_fr
        ax_raster.sharex(bottom_ax)
        ax_fr.sharex(bottom_ax)
        ax_raster.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        ax_fr.xaxis.set_major_locator(MaxNLocator(integer=True))
        if show_decoder:
            ax_fr.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
        if row == nrows - 1:
            bottom_ax.set_xlabel('Time from stimulus onset (s)')
        ax_raster.set_title(f'Trial {trial}', fontsize=12)
        sns.despine(ax=ax_raster, bottom=True)
        sns.despine(ax=ax_fr, bottom=True if show_decoder else False)
        if show_decoder:
            sns.despine(ax=ax_decoder)

    # Hide unused cells
    for idx in range(n_trials, nrows * ncols):
        row = idx // ncols
        col = idx % ncols
        for k in range(n_subrows):
            axes[row * n_subrows + k, col].axis('off')

    fig.tight_layout()
    if save_path:
        plt.savefig(save_path + filename_base + '_live_grid.svg', format='svg', dpi=300)
    plt.show()
    plt.close()

In [ ]:
# Sample 10 neurons per population
neurons = (
    dfSP_raster.groupby('population')['neuron']
    .apply(lambda x: pd.Series(x.unique()).sample(10))
    .reset_index(drop=True)
)
dfSP_low = dfSP_raster[dfSP_raster.neuron.isin(neurons)]

plot_trials_grid_live(
    dfSP=dfSP_low, dfLL=dfLL_raster,
    show_decoder=True, trials_list=[6, 7],
    align='Stimulus_ON', ncols=4,
    filename_base='sample_neurons',
    save_path=save_figures,
)

In [ ]:
# All neurons, different trials
plot_trials_grid_live(
    dfSP=dfSP_raster, dfLL=dfLL_raster,
    trials_list=[8, 9], align='Stimulus_ON',
    show_decoder=True, ncols=4,
    filename_base='all_neurons',
    save_path=save_figures,
)

---
## 2. Log-odds distributions â€” Mongillo model

Requires `mongillo_model_LLs_{win}ms.csv` with columns `iter`, `stimulus`, `loglikelihood`.
If the file uses different column names (e.g. `trial`, `choice`) adjust the rename line below.

In [ ]:
plot = pd.read_csv(data_path + f'decoder_all_mongillo_window{win}_20_r{r}.csv', index_col=0)

In [ ]:
# Single-iteration log-odds histogram
split_by_choice = True
bins        = np.linspace(-5, 5, 60)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

choices = np.sort(plot['stimulus'].unique())
it      = plot['iter'].unique()[6]

fig, ax = plt.subplots(figsize=(5, 4))
sdata = plot[plot['iter'] == it]

for choice in choices:
    cdata = sdata.loc[sdata['stimulus'] == choice, 'loglikelihood'].dropna()
    hist, _ = np.histogram(cdata, bins=bins, density=True)
    line = ax.plot(bin_centers, hist, linewidth=1)
    color = line[0].get_color()
    ck = kurtosis(cdata, fisher=True, bias=False)
    cs = skew(cdata, bias=False)
    ax.text(0.02, 0.5 + 0.1 * choice,
            f'k={ck:.2f}, sk={cs:.2f}',
            transform=ax.transAxes, color=color, fontsize=8, ha='left', va='top')

ax.set_title(f'iter {it}', fontsize=10)
ax.set_xlabel('Log odds')
ax.set_ylabel('Density')
ax.set_xlim(-5, 5)
plt.yticks([0, 0.5, 1], ['0', '0.5', '1'])
sns.despine()
plt.tight_layout()
plt.savefig(save_figures + f'Log_odds_iter{it}_window{r}_network{win}.svg', format='svg')
plt.show()

In [ ]:
# Aggregated log-odds by choice across all iterations
split_by_choice = True
ranges = 3

plot['log_odds_norm'] = (
    plot.groupby(['iter'])['loglikelihood']
        .transform(lambda x: (x - x.mean()) / x.std(ddof=0))
)

bins        = np.linspace(-ranges, ranges, ranges * 20)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
sessions    = np.sort(plot['iter'].unique())
choices     = np.sort(plot['stimulus'].unique())

hist_by_choice = {c: [] for c in choices}
k_by_choice    = {c: [] for c in choices}
sk_by_choice   = {c: [] for c in choices}

for session in sessions:
    sdata = plot[plot['iter'] == session]
    for c in choices:
        vals = sdata.loc[sdata['stimulus'] == c, 'log_odds_norm'].dropna()
        hist, _ = np.histogram(vals, bins=bins, density=True)
        hist_by_choice[c].append(hist)
        k_by_choice[c].append(kurtosis(vals, fisher=True, bias=False))
        sk_by_choice[c].append(skew(vals, bias=False))

plt.figure(figsize=(5, 4))
for c in choices:
    mean_hist = np.mean(hist_by_choice[c], axis=0)
    mean_k    = np.mean(k_by_choice[c])
    mean_sk   = np.mean(sk_by_choice[c])
    line  = plt.plot(bin_centers, mean_hist, linewidth=2, label=f'choice {c}')
    color = line[0].get_color()
    plt.text(0.5, 0.95 - 0.24 * c,
             f'{c}: k={mean_k:.2f},\n sk={mean_sk:.2f}',
             transform=plt.gca().transAxes, color=color, va='top')

plt.xlabel('Z-scored log odds')
plt.ylabel('Density')
plt.xticks([-ranges, 0, ranges])
plt.ylim(-0.05, 1.1)
plt.yticks([0, 0.5, 1], ['0', '0.5', '1'])
plt.legend().remove()
sns.despine()
plt.tight_layout()
plt.savefig(save_figures + f'Log_odds_per_choice_aggregated_session_window{r}_network{win}.svg', format='svg')
plt.show()

---
## 3. Log-odds distributions â€” neural data

Requires `ephys_logodds.csv` with columns: `delay`, `times`, `log_odds`, `choice`, `session`, `trial_type`.

In [ ]:
df_cum_sti = pd.read_csv(data_path + 'decoder_results_log_odds_100r_withRL_L2.csv', index_col=0)

# Define delay periods: delay_value â†’ (t_start, t_end) in seconds
periods    = {10.0: (0.5, 9.5)}   # adjust as needed
trial_order = df_cum_sti['trial_type'].unique().tolist()  # or set manually e.g. ['WM', 'RL']
sessions   = np.sort(df_cum_sti['session'].unique())

In [ ]:
# Extract delay-period time windows â†’ plot_neurons
frames = []
for delay, (t0, t1) in periods.items():
    df_period = df_cum_sti.loc[
        (df_cum_sti['delay'] == delay) &
        (df_cum_sti['times'] >= t0) &
        (df_cum_sti['times'] <= t1)
    ].copy()
    frames.append(df_period)

plot_neurons = pd.concat(frames, ignore_index=True)

In [ ]:
# Per-session log-odds histogram grid
split_by_choice = True
sessions = np.sort(plot_neurons['session'].unique())
n_sessions = len(sessions)
ncols = 4
nrows = int(np.ceil(n_sessions / ncols))
choices = np.sort(plot_neurons['vector_answer'].unique())

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=False)
axes = axes.flatten()

for ax, session in zip(axes, sessions):
    sdata = plot_neurons[plot_neurons['session'] == session]
    session_name = sdata.session_name.unique()[0]  # Assuming session_name is consistent within a session
    if split_by_choice:
        for choice in choices:
            cdata = sdata[sdata['vector_answer'] == choice]['real'].dropna()
            bins_s = np.linspace(cdata.quantile(0.01), cdata.quantile(0.99), 30)
            bin_centers_s = 0.5 * (bins_s[:-1] + bins_s[1:])
            hist, _ = np.histogram(cdata, bins=bins_s, density=True)
            ax.plot(bin_centers_s, hist, linewidth=1, label=f'choice {choice}')
    else:
        vals = sdata['real'].dropna()
        bins_s = np.linspace(vals.quantile(0.01), vals.quantile(0.99), 30)
        bin_centers_s = 0.5 * (bins_s[:-1] + bins_s[1:])
        hist, _ = np.histogram(vals, bins=bins_s, density=True)
        ax.fill_between(bin_centers_s, hist, step='mid', alpha=0.4, color='grey')

    abs_max = max(abs(sdata['real'].dropna().quantile(0.01)),
                  abs(sdata['real'].dropna().quantile(0.99)))
    abs_max = np.ceil(abs_max)
    ax.set_xlim(-abs_max, abs_max)
    ax.set_xticks([-abs_max, 0, abs_max])
    ax.set_ylim(bottom=0)
    y_max = np.ceil(ax.get_ylim()[1] * 10) / 10
    ax.set_ylim(top=y_max)
    ax.set_yticks([0, y_max / 2, y_max])
    ax.set_title(f'{session_name}')
    ax.set_xlabel('Z-scored log odds')
    ax.set_ylabel('Density')

for ax in axes[n_sessions:]:
    ax.axis('off')

sns.despine()
plt.tight_layout()
plt.savefig(save_figures + 'Log_odds_per_session.svg', format='svg')
plt.show()

In [ ]:
# Aggregated log-odds by choice across sessions
split_by_choice = True

plot_neurons['log_odds_norm'] = (
    plot_neurons.groupby(['session'])['real']
        .transform(lambda x: (x - x.mean()) / x.std(ddof=0))
)
plot_neurons = plot_neurons[plot_neurons['trial_type'].isin(trial_order)]

bins        = np.linspace(-4, 4, 40)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
sessions    = np.sort(plot_neurons['session'].unique())
choices     = np.sort(plot_neurons['vector_answer'].unique())

hist_by_choice_n = {c: [] for c in choices}
k_by_choice_n    = {c: [] for c in choices}
sk_by_choice_n   = {c: [] for c in choices}

for session in sessions:
    sdata = plot_neurons[plot_neurons['session'] == session]
    for c in choices:
        vals = sdata.loc[sdata['vector_answer'] == c, 'log_odds_norm'].dropna()
        hist, _ = np.histogram(vals, bins=bins, density=True)
        hist_by_choice_n[c].append(hist)
        k_by_choice_n[c].append(kurtosis(vals, fisher=True, bias=False))
        sk_by_choice_n[c].append(skew(vals, bias=False))

plt.figure(figsize=(5, 4))
for c in choices:
    mean_hist = np.mean(hist_by_choice_n[c], axis=0)
    mean_k    = np.mean(k_by_choice_n[c])
    mean_sk   = np.mean(sk_by_choice_n[c])
    line  = plt.plot(bin_centers, mean_hist, linewidth=2, label=f'choice {c}')
    color = line[0].get_color()
    plt.text(0.65, 0.95 - 0.24 * c,
             f'{c}: k={mean_k:.2f},\n sk={mean_sk:.2f}',
             transform=plt.gca().transAxes, color=color, va='top')

plt.xlabel('Z-scored log odds')
plt.ylabel('Density')
plt.xticks([-4, 0, 4])
plt.ylim(-0.05, 1)
plt.yticks([0, 0.5, 1], ['0', '0.5', '1'])
plt.legend().remove()
sns.despine()
plt.tight_layout()
plt.savefig(save_figures + f'Log_odds_per_choice_{split_by_choice}_aggregated_session.svg', format='svg')
plt.show()

---
## 4. Bimodality coefficient

`plot_neurons` (neural) must have `log_odds_norm` â€” run section 3 first.  
`plot` (model) is renamed to `plot_mongillo` with session/log_odds columns.

In [ ]:
# Build plot_mongillo from model log-odds with compatible column names
plot_mongillo = plot.rename(columns={'iter': 'session', 'loglikelihood': 'log_odds'}).copy()
plot_mongillo['log_odds_norm'] = (
    plot_mongillo.groupby('session')['log_odds']
        .transform(lambda x: (x - x.mean()) / x.std(ddof=0))
)
# Rename stimulus â†’ choice if needed
if 'stimulus' in plot_mongillo.columns and 'choice' not in plot_mongillo.columns:
    plot_mongillo = plot_mongillo.rename(columns={'stimulus': 'vector_answer'})

bc_results  = {}
bc_sessions = {}

for label, df_bc in [('sample2', plot_mongillo), ('sample1', plot_neurons)]:
    all_bc   = []
    sessions_bc = np.sort(df_bc['session'].unique())
    for session in sessions_bc:
        sdata = df_bc[df_bc['session'] == session]
        vals  = sdata.apply(
            lambda row: -row['log_odds_norm'] if row['vector_answer'] == 0 else row['log_odds_norm'],
            axis=1
        ).dropna()
        ck = kurtosis(vals, fisher=False, bias=False)
        cs = skew(vals, bias=False)
        all_bc.append((cs ** 2 + 1) / ck)
    bc_sessions[label] = all_bc
    bc_results[label]  = np.mean(all_bc)

print(bc_results)

fig, ax = plt.subplots(figsize=(3.5, 4))

x1 = 0
sns.boxplot(
    x=[x1] * len(bc_sessions['sample1']),
    y=bc_sessions['sample1'], ax=ax,
    showfliers=False, width=0.4, color='darkgrey',
    medianprops=dict(color='white', linewidth=1),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=0),
    boxprops=dict(edgecolor='none'),
)
jitter = np.random.uniform(-0.08, 0.08, size=len(bc_sessions['sample1']))
ax.scatter(x1 + jitter, bc_sessions['sample1'], color='black', s=20, zorder=3, alpha=0.6)

x2 = 1
ax.errorbar(x2, np.mean(bc_sessions['sample2']),
            yerr=np.std(bc_sessions['sample2']),
            fmt='D', color='red', markersize=6, zorder=3, capsize=3)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Neurons', 'Mongillo'], fontsize=14)
ax.set_yticks([0, 0.5, 1])
ax.set_yticklabels(['0', '0.5', '1'])
ax.set_ylabel('Bimodality coefficient', fontsize=14)
ax.set_xlabel('')
ax.hlines(5 / 9, -0.5, 1.5, color='grey', linestyle='--', linewidth=1)
ax.set_ylim(bottom=0)
sns.despine()
plt.tight_layout()
plt.savefig(save_figures + 'Bimodality_coefficient_comparison.svg', format='svg')
plt.show()

---
## 5. Autocorrelogram (ACG)

Loads precomputed `.npz` files. Run section 7 first if the files don't exist.

### 5a. Mongillo model ACG

In [ ]:
data_acg_m = np.load(data_path + f'mongillo_acg_{win}ms.npz', allow_pickle=True)
acg_mat_m  = data_acg_m['acg_mat']   # shape: (n_neurons, n_lags)
lags_m     = data_acg_m['lags']

n_remove   = 1
center_idx = np.argmin(np.abs(lags_m))

# Remove zero-lag artifact
mean_per_neuron_nan = acg_mat_m.copy()
mean_per_neuron_nan[:, center_idx - n_remove: center_idx + n_remove + 1] = np.nan

# Bootstrap
n_boot          = 1000
n_neurons_boot  = 20
n_neurons_total = mean_per_neuron_nan.shape[0]

boot_grand_means = []
for _ in range(n_boot):
    idx = np.random.choice(n_neurons_total, size=n_neurons_boot, replace=True)
    boot_grand_means.append(np.nanmean(mean_per_neuron_nan[idx], axis=0))

boot_grand_means = np.vstack(boot_grand_means)
grand_mean = np.nanmean(boot_grand_means, axis=0)
ci_lo, ci_hi = np.nanpercentile(boot_grand_means, [2.5, 97.5], axis=0)

plt.figure(figsize=(5, 4.5))
plt.fill_between(lags_m, ci_lo, ci_hi, alpha=0.3, color='black', edgecolor='none')
plt.plot(lags_m, grand_mean, color='black', linewidth=2)
plt.xlabel('Lag (s)')
plt.ylabel('Session average\nACG')
plt.yticks([0., 1.0, 2, 3])
plt.ylim(0., 3)
sns.despine()
plt.tight_layout()
plt.savefig(save_figures + f'ACG_bootstrapped_mongillo_window{win}.svg', format='svg')
plt.show()

### 5b. Neural data ACG

In [ ]:
data_acg_e        = np.load(data_path + 'ephys_acg.npz', allow_pickle=True)
session_means_mat = data_acg_e['session_means_mat']   # (n_sessions, n_lags)
session_keys      = data_acg_e['session_keys']        # session file names
lags_no0          = data_acg_e['lags_no0']            # lags with zero removed
acg_mat_e         = data_acg_e['acg_mat']             # (n_neurons_all_sessions, n_lags)
lags_e            = data_acg_e['lags']

session_means_no0 = {
    k: session_means_mat[i]
    for i, k in enumerate(session_keys)
}

In [ ]:
# Per-session ACG grid (random subset)
session_exclude = [
    'E22_2022-01-15_16-53-52.csv', 'E04_2021-03-30_11-20-16.csv',
    'E13_2021-05-25_16-26-57.csv', 'E19_2022-01-17_15-34-37.csv',
    'E13_2021-06-09_12-14-21.csv'
]

n_sessions_show = 24
n_cols = 4
all_sessions = [s for s in list(session_means_no0.keys()) if s not in session_exclude]
sessions_show = np.random.choice(all_sessions,
                                  min(n_sessions_show, len(all_sessions)), replace=False)
n_rows = int(np.ceil(len(sessions_show) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(4 * n_cols, 4 * n_rows),
                          sharex=True, sharey=True)
axes = axes.flatten()

for ax, session in zip(axes, sessions_show):
    ax.plot(lags_no0, session_means_no0[session], color='black', linewidth=2)
    ax.axhline(0, linestyle='--', color='gray', linewidth=1)
    ax.set_title(session, fontsize=9)

for ax in axes[len(sessions_show):]:
    ax.axis('off')

axes[0].set_ylabel('Session mean ACG')
for ax in axes[-n_cols:]:
    ax.set_xlabel('Lag (s)')

plt.ylim(0.25, 3)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Grand average ACG â€” neural data (session_means_mat already has zero-lag removed)

grand_mean_e = np.nanmean(session_means_mat, axis=0)
grand_sem_e  = np.nanstd(session_means_mat, axis=0) / np.sqrt(session_means_mat.shape[0])

idx_show = np.random.choice(len(session_keys), min(5, len(session_keys)), replace=False)

plt.figure(figsize=(6, 4))
for i in idx_show:
    plt.plot(lags_no0, session_means_mat[i], alpha=0.5)

plt.plot(lags_no0, grand_mean_e, color='black', linewidth=2)
plt.fill_between(lags_no0,
                 grand_mean_e - grand_sem_e,
                 grand_mean_e + grand_sem_e,
                 color='black', alpha=0.2, edgecolor='none')
plt.xlabel('Lag (s)')
plt.ylabel('ACG')
plt.hlines(0, lags_no0[0], lags_no0[-1], colors='gray', linestyles='dashed', linewidth=1)
sns.despine()
plt.tight_layout()
plt.savefig(save_figures + 'grand_average_ACG_sessions.svg', format='svg')
plt.show()


---
## 6. Cross-correlogram (CCG)

Loads precomputed `.npz` files. Run section 7 first if the files don't exist.

### 6a. Mongillo model CCG

In [ ]:
data_ccg_m      = np.load(data_path + f'mongillo_ccg_{win}ms.npz', allow_pickle=True)
same_sign_array = data_ccg_m['same_sign_mat']   # (n_pairs, n_lags)
opp_mixed_array = data_ccg_m['opp_mixed_mat']
lags_ccg_m      = data_ccg_m['lags']

def bootstrap_ccg(ccg_array, n_boot=1000):
    if ccg_array is None or len(ccg_array) == 0:
        return None
    n_pairs    = ccg_array.shape[0]
    boot_means = []
    for _ in range(n_boot):
        idx = np.random.choice(n_pairs, size=n_pairs, replace=True)
        boot_means.append(ccg_array[idx].mean(axis=0))
    return np.vstack(boot_means)

def get_mean_ci(boot_array):
    if boot_array is None:
        return None, None, None
    return (boot_array.mean(axis=0),
            np.percentile(boot_array, 2.5,  axis=0),
            np.percentile(boot_array, 97.5, axis=0))

same_sign_boot = bootstrap_ccg(same_sign_array)
opp_mixed_boot = bootstrap_ccg(opp_mixed_array)

conditions = [('Within', same_sign_boot), ('Across', opp_mixed_boot)]
fig, axes  = plt.subplots(1, 2, figsize=(9, 5), sharex=True, sharey=True)

for ax, (title, boot_array) in zip(axes, conditions):
    mean_ccg, lower_ci, upper_ci = get_mean_ci(boot_array)
    if mean_ccg is None:
        ax.text(0.5, 0.5, f'No data for\n{title}', ha='center', va='center')
        continue
    ax.plot(lags_ccg_m, mean_ccg, color='black', linewidth=2)
    ax.fill_between(lags_ccg_m, lower_ci, upper_ci, color='gray', alpha=0.3)
    ax.set_title(title)
    ax.set_xlabel('Lag (s)')
    # ax.set_ylim(0.5, 1)
    # ax.set_yticks([0.5, 0.75, 1])
    sns.despine()

axes[0].set_ylabel('Bootstrap mean CCG')
plt.tight_layout()
plt.savefig(save_figures + f'CCG_by_weight_sign_mongillo_window{win}.svg', format='svg')
plt.show()

### 6b. Neural data CCG

In [ ]:
data_ccg_e  = np.load(data_path + 'ephys_ccg.npz', allow_pickle=True)
same_mat_e  = data_ccg_e['same_sign_mat']   # stacked: (total_trials, n_lags)
opp_mat_e   = data_ccg_e['opp_mixed_mat']
lags_ccg_e  = data_ccg_e['lags']

def stack_and_get_mean(ccg_mat):
    if ccg_mat is None or len(ccg_mat) == 0:
        return None, None
    mean_ac = ccg_mat.mean(axis=0)
    sem_ac  = ccg_mat.std(axis=0) / np.sqrt(ccg_mat.shape[0])
    return mean_ac, sem_ac

conditions = [('Within', same_mat_e), ('Across', opp_mat_e)]
fig, axes  = plt.subplots(1, 2, figsize=(9, 5), sharex=True, sharey=True)

for ax, (title, ccg_mat) in zip(axes, conditions):
    mean_ac, sem_ac = stack_and_get_mean(ccg_mat)
    if mean_ac is None:
        ax.text(0.5, 0.5, f'No data for {title}', ha='center', va='center')
        continue
    ax.plot(lags_ccg_e, mean_ac, color='black')
    ax.fill_between(lags_ccg_e,
                    mean_ac - sem_ac * 1.96,
                    mean_ac + sem_ac * 1.96,
                    alpha=0.3, color='gray')
    ax.set_title(title)
    ax.set_xlabel('Lag (s)')
    ax.set_ylim(0.5, 3)
    ax.set_yticks([0, 1, 2, 3])
    sns.despine()

axes[0].set_ylabel('Session average\nCCG')
plt.tight_layout()
plt.savefig(save_figures + 'CCG_by_weight_sign.svg', format='svg')
plt.show()